# Proyecto Grupal Integrador

**Licenciatura en Ciencia de Datos**
<br>
<strong>Materia:</strong> Ciencia de Datos
<br>
<strong>Docentes:</strong> German Giró, Malena Constancio y Giselle Gustamante
<br>
<strong>Integrantes:</strong> Hector Maximiliano Garcia Perez, Julian Alejandro Siles, Lourdes Carolina Benitez Garay y Franco Damián Vicente

 ---

# Entrega 2: Análisis exploratorio y visualización

## A. Preparación de los datos

En esta sección se aplicarán las técnicas de data wrangling solicitadas para la correcta realizanción del segundo entregable.

### Carga del dataset

#### Importación de librerias

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### Inicialización y configuración del versionado

Creación de la estructura de directorios local necesaria para almacenar los datasets crudos, intermedios y procesados, junto con los gráficos generados.

In [ ]:
def setup_project_structure() -> None:
  """
  Crea la jerarquía de directorios.
  
  Se utiliza os.makedirs con exist_ok=True para evitar errores si ya existen
  los directorios.
  
  Args:
    Ninguno.
    
  Returns:
    None
  """

  directorios = [
    "icu-patient-analysis-2014-15/data/raw",
    "icu-patient-analysis-2014-15/data/interim/plots",
    "icu-patient-analysis-2014-15/data/processed"
  ]
  
  for directorio in directorios:
    os.makedirs(directorio, exist_ok=True)
    print(f"Directorio verificado: {directorio}")

setup_project_structure()

#### Carga y Exploración Inicial de Datos

Descarga del dataset original en formato CSV desde un archivo de google drive accesible mediante link. Almacenamiento local en el directorio <code>raw</code> para evitar dependencias externas.

In [ ]:
def load_raw_data() -> pd.DataFrame:
  """
  Descarga el dataset desde la URL de la cátedra y lo almacena localmente.
  
  Returns:
    pd.DataFrame: El DataFrame con los datos crudos cargados.
  """

  # ID del archivo en Google Drive
  file_id: str = "1KI45o0t-b3MtRa1QNBuZN7AL7F57Xp0X"

  # URL de descarga directa de Drive
  url_dataset: str = f"https://drive.google.com/uc?export=download&id={file_id}"

  ruta_local: str = "icu-patient-analysis-2014-15/data/raw/patient.csv"
  
  df_raw = pd.read_csv(url_dataset)
  
  df_raw.to_csv(ruta_local, index=False)
  print(f"Dataset descargado y guardado en: {ruta_local}")
  
  return df_raw

df = load_raw_data()

### A.1. Tratamiento de datos faltantes

In [ ]:
def handle_missing_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Realiza el tratamiento de datos faltantes aplicando eliminación por lista 
    (Listwise Deletion) para variables críticas y descartando columnas irrelevantes.
    
    Args:
      df (pd.DataFrame): DataFrame con los datos crudos.
      
    Returns:
      pd.DataFrame: DataFrame limpio de datos faltantes (NaN/Nulls).
    """
    
    # 1. Eliminación de columnas irrelevantes
    # Se descartan porque no aportan a las preguntas del proyecto o tienen demasiados nulos.
    cols_to_drop = [
        'dischargeweight', 'hospitaladmitsource', 'apacheadmissiondx', 
        'ethnicity', 'hospitaldischargelocation', 'unitadmitsource', 
        'hospitaldischargestatus', 'unitdischargelocation'
    ]
    # Usamos errors='ignore' por si alguna columna ya no existe en el DataFrame
    df_cleaned = df.drop(columns=cols_to_drop, errors='ignore')
    
    # 2. Eliminación de filas con datos nulos en variables críticas
    # Se eliminan filas donde falte la edad, altura, peso, género o estado de alta.
    critical_cols = ['age', 'admissionheight', 'admissionweight', 'gender', 'unitdischargestatus']
    df_cleaned = df_cleaned.dropna(subset=critical_cols)
    
    # 3. Guardado en la estructura de directorios
    # Se guarda en interim ya que es un paso intermedio antes de la transformación de tipos
    ruta_interim: str = "icu-patient-analysis-2014-15/data/interim/patient_interim.csv"
    
    # Nos aseguramos de que el directorio base exista
    os.makedirs(os.path.dirname(ruta_interim), exist_ok=True)
    
    # Exportamos a CSV sin el índice para mantenerlo limpio
    df_cleaned.to_csv(ruta_interim, index=False)
    print(f"Dataset sin valores nulos guardado en: {ruta_interim}")
    
    return df_cleaned

# Ejecución de la limpieza de nulos
df_interim = handle_missing_data(df)

# Reporte del impacto de la estrategia
print("-" * 40)
print(f"Dimensiones originales: {df.shape}")
print(f"Dimensiones luego de eliminar nulos: {df_interim.shape}")
print(f"Total de filas eliminadas: {df.shape[0] - df_interim.shape[0]}")

In [ ]:
# Verificación de valores faltantes en el DataFrame interim

print("-" * 40)
print("Verificación de valores faltantes en df_interim")

# Cantidad total de valores faltantes en todo el DataFrame
total_missing = df_interim.isna().sum().sum()

print(f"Total de valores faltantes en df_interim: {total_missing}")

# Cantidad de valores faltantes por columna
missing_by_column = df_interim.isna().sum()

# Mostrar solo columnas que todavía tienen valores faltantes
columns_with_missing = missing_by_column[missing_by_column > 0]

if columns_with_missing.empty:
    print("No se encontraron valores faltantes en df_interim.")
else:
    print("Columnas con valores faltantes:")
    print(columns_with_missing)

#### Justificación de la estrategia de tratamiento de datos faltantes

La estrategia adoptada fue la eliminación por lista (Listwise Deletion) para los registros incompletos y la eliminación de características (Feature dropping) para las columnas no relevantes. Al separar el proceso, garantizamos que las 224 filas eliminadas obedecen exclusivamente a la falta de información empírica en variables críticas para el estudio (como peso, altura o sexo), salvaguardando un total de 2296 observaciones completas.

### A.2. Corrección de formatos y tipos de datos

In [ ]:
def correct_data_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    Corrige los tipos de datos del DataFrame, adaptando la variable 'age'
    de formato texto a numérico para permitir su análisis estadístico.
    
    Args:
      df (pd.DataFrame): DataFrame intermedio (previamente sin nulos).
      
    Returns:
      pd.DataFrame: DataFrame con el tipo de dato de 'age' corregido.
    """
    # Creamos una copia para no alterar el DataFrame original accidentalmente
    df_copy = df.copy()
    
    # --- Corrección de la variable 'age' ---
    # Reemplazamos el texto '> 89' por '90'
    df_copy['age'] = df_copy['age'].replace('> 89', '90')
    
    # Convertimos la columna de texto (str/object) a número entero (int)
    df_copy['age'] = df_copy['age'].astype(int)
    
    return df_copy

# Ejecutamos la corrección sobre el dataset intermedio que obtuvimos en el paso anterior
df_interim = correct_data_types(df_interim)

# Verificamos los cambios
print("Tipo de dato actualizado en la columna 'age':")
print(df_interim['age'].dtype)

#### Justificación de la correción de formato en age

Se detectó que la columna age estaba tipada como texto debido a la presencia del valores coficados como "> 89", causados por metodos de anonimización de pacientes. Para poder responder a la Pregunta 3 planteada en la Entrega 1 (estratificación del IMC por grupos de edad), es indispensable que la edad sea una variable cuantitativa discreta y operable matemáticamente.

Por lo tanto, se decidio recodificar dichos valores como "90" y posteriormente forzar la conversión de la columna age a un tipo de dato numérico (int) para permitir su análisis estadístico.

### A.3. Limpieza de datos textuales

In [ ]:
def clean_text_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Estandariza las columnas de texto categórico convirtiendo todos los 
    caracteres a minúsculas y eliminando espacios en blanco accidentales 
    (al inicio y al final de cada cadena).
    
    Args:
      df (pd.DataFrame): DataFrame intermedio.
      
    Returns:
      pd.DataFrame: DataFrame con los datos de texto limpios.
    """
    df_copy = df.copy()
    
    # Definimos las columnas textuales a limpiar
    text_cols = ['gender', 'unittype', 'unitstaytype', 'unitdischargestatus']
    
    for col in text_cols:
        if col in df_copy.columns:
            # 1. str.strip(): Elimina espacios en blanco iniciales y finales.
            # 2. str.lower(): Convierte todo a minúsculas.
            df_copy[col] = df_copy[col].str.strip().str.lower()
            
    return df_copy

# Ejecutamos la limpieza textual
df_interim = clean_text_data(df_interim)

# Verificamos los resultados en la columna clave para la Pregunta 2
print("Valores únicos en 'gender' tras la limpieza:")
print(df_interim['gender'].unique())

print("\nValores únicos en 'unitdischargestatus' tras la limpieza:")
print(df_interim['unitdischargestatus'].unique())

#### Justificación de la limpieza
La consistencia de las variables categóricas es fundamental para evitar la fragmentación de los grupos de estudio. Debido a esto, decidimos proceder a aplicando técnicas de normalización sobre las columnas textuales (gender, unittype, unitstaytype, y unitdischargestatus).

Mediante el uso encadenado de los métodos <code>.str.strip()</code> y <code>.str.lower()</code>, garantizamos que todas las cadenas de texto estén libres de espacios en blanco espurios y que posean una capitalización uniforme (minúsculas). Por ejemplo, valores como "Female", " FEMALE" o "female " ahora convergen unificadamente en "female". Esta estandarización es un paso preventivo que asegura la precisión en las futuras etapas de agrupación, siendo especialmente crítico para la variable gender, la cual es indispensable para resolver la Pregunta 2 planteada en la Entrega 1 (modificación de la relación IMC-estancia según el sexo del paciente).

### A.4. Transformaciones necesarias

En esta sección procedemos a crear las variables analíticas que responderán directamente a las preguntas de investigación. Iniciamos creando una copia de los datos intermedios y aplicando un filtro de calidad para descartar registros fisiológicamente implausibles en peso y altura, asegurando la validez del cálculo del Índice de Masa Corporal.

In [ ]:
# Creamos la base para procesar a partir de los datos intermedios (textos y nulos limpios)
df_processed = df_interim.copy()

# 1. Filtrado de ruido en variables físicas
mask_valid_physique = (
    (df_processed['admissionheight'] >= 100) & (df_processed['admissionheight'] <= 250) &
    (df_processed['admissionweight'] >= 30) & (df_processed['admissionweight'] <= 300)
)
df_processed = df_processed[mask_valid_physique].copy()

#### Cálculo y categorización del Índice de Masa Corporal (IMC)
A partir del peso y la altura validados, se calcula el IMC continuo. Posteriormente, se operacionaliza la variable en una nueva columna categórica (<code>imc_category</code>) utilizando los umbrales oficiales de la Organización Mundial de la Salud (OMS), lo cual es fundamental para comparar la estancia en UCI entre pacientes normopeso y obesos.

In [ ]:
# 2. Creación de variables de IMC
# Fórmula: peso (kg) / altura (m)^2
df_processed['imc'] = df_processed['admissionweight'] / ((df_processed['admissionheight'] / 100) ** 2)

# Discretización en categorías de la OMS
bins_imc = [0, 18.5, 25.0, 30.0, np.inf]
labels_imc = ['Bajo peso', 'Normal', 'Sobrepeso', 'Obesidad']

# right=False asegura que los intervalos sean inclusivos a la izquierda y exclusivos a la derecha [x, y)
df_processed['imc_category'] = pd.cut(
    df_processed['imc'], 
    bins=bins_imc, 
    labels=labels_imc, 
    right=False
)

#### Operacionalización del tiempo de estancia y estratificación etaria
Para facilitar la interpretación de los modelos estadísticos y visualizaciones posteriores, se transforma la variable de estancia en UCI de minutos a días. Además, se segmenta la variable continua de edad en tres estratos clínicos (Jóvenes, Adultos, Ancianos) para evaluar si el efecto del IMC varía según la reserva fisiológica del grupo etario.

In [ ]:
# 3. Transformación temporal de la Estancia en UCI
# Convertimos el offset (minutos) a días (1 día = 1440 minutos)
df_processed['icu_los_days'] = df_processed['unitdischargeoffset'] / 1440.0

# 4. Estratificación por grupos etarios
# Cortes: Jóvenes (0-39), Adultos (40-64), Ancianos (65+)
bins_age = [0, 39, 64, 150]
labels_age = ['Jóvenes', 'Adultos', 'Ancianos']

df_processed['age_group'] = pd.cut(
    df_processed['age'], 
    bins=bins_age, 
    labels=labels_age
)

#### Exportación del dataset procesado
Finalmente, el dataset consolidado se exporta a la carpeta `processed`. Al haber depurado el ruido y optimizado los tipos de datos, este archivo liviano servirá como la "fuente de la verdad" para la etapa de análisis descriptivo y visualización del equipo.

In [ ]:
# 5. Exportación del Dataset
ruta_processed = "icu-patient-analysis-2014-15/data/processed/patient_processed.csv"

# Aseguramos la existencia del directorio base
os.makedirs(os.path.dirname(ruta_processed), exist_ok=True)

# Exportamos sin índice
df_processed.to_csv(ruta_processed, index=False)

# Reporte final
print(f"Dataset procesado final guardado en: {ruta_processed}")
print(f"Dimensiones finales para análisis: {df_processed.shape}")